## Module C — Visualization & insights (**daily**, corrected)\n\nThis notebook updates the visualization module to use **daily aligned data** (not annual means):\n\n- Dual-axis **normalized price** trends (GLD vs NFLX)\n- Scatter of **daily returns** with regression fit line\n- Correlation heatmap (returns)\n- Rolling correlation (to show dependence changes over time)\n- Tail-day highlighting (worst 5% NFLX days)\n\nInput files (from Module A daily or the script output):\n- `outputs/combined_daily_prices.csv`\n- `outputs/combined_daily_logreturns.csv`\n

In [ ]:
from pathlib import Path
\nimport numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
\nROOT = Path.cwd()
OUT_DIR = ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

prices_path = OUT_DIR / "combined_daily_prices.csv"
rets_path = OUT_DIR / "combined_daily_logreturns.csv"

prices = pd.read_csv(prices_path)
rets = pd.read_csv(rets_path)
prices["Date"] = pd.to_datetime(prices["Date"])
rets["Date"] = pd.to_datetime(rets["Date"])

prices.head(), rets.head()


### Dual-axis normalized price trends\n\nBecause NFLX and GLD are on different price scales, we normalize each series to 1.0 at the first date.\n

In [ ]:
sns.set_style("whitegrid")\np = prices.sort_values("Date").copy()\np["NFLX_norm"] = p["NFLX_AdjClose"] / p["NFLX_AdjClose"].iloc[0]\np["GLD_norm"] = p["GLD_NAV"] / p["GLD_NAV"].iloc[0]\n\nfig, ax1 = plt.subplots(figsize=(12, 5))\nax1.plot(p["Date"], p["GLD_norm"], color="tab:gold", linewidth=1.8, label="GLD (normalized)")\nax1.set_ylabel("GLD normalized", color="tab:gold")\nax1.tick_params(axis="y", labelcolor="tab:gold")\n\nax2 = ax1.twinx()\nax2.plot(p["Date"], p["NFLX_norm"], color="tab:blue", linewidth=1.8, label="NFLX (normalized)")\nax2.set_ylabel("NFLX normalized", color="tab:blue")\nax2.tick_params(axis="y", labelcolor="tab:blue")\n\nfig.suptitle("Daily normalized trend: GLD vs NFLX")\nfig.tight_layout()\nout = FIG_DIR / "dual_axis_trend_daily_normalized.png"\nfig.savefig(out, dpi=200, bbox_inches="tight")\nout\n

### Scatter of daily returns + regression fit line\n

In [ ]:
x = rets["GLD_logret"].to_numpy()\ny = rets["NFLX_logret"].to_numpy()\nX = sm.add_constant(x)\nres = sm.OLS(y, X, missing="drop").fit(cov_type="HC1")\n\ngrid = np.linspace(np.nanmin(x), np.nanmax(x), 200)\nyhat = res.params[0] + res.params[1] * grid\n\nfig, ax = plt.subplots(figsize=(7.5, 5.5))\nax.scatter(x, y, s=10, alpha=0.25)\nax.plot(grid, yhat, color="crimson", linewidth=2, label=f"OLS (HC1) slope={res.params[1]:.4f}")\nax.set_xlabel("GLD daily log return")\nax.set_ylabel("NFLX daily log return")\nax.set_title("Daily returns scatter (with OLS fit)")\nax.legend()\nfig.tight_layout()\nout = FIG_DIR / "scatter_returns_with_fit_daily.png"\nfig.savefig(out, dpi=200, bbox_inches="tight")\nout\n

### Correlation heatmap (returns)\n

In [ ]:
corr = rets[["NFLX_logret", "GLD_logret"]].corr()\nfig, ax = plt.subplots(figsize=(5, 4))\nsns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)\nax.set_title("Correlation heatmap (daily log returns)")\nfig.tight_layout()\nout = FIG_DIR / "correlation_heatmap_returns_daily.png"\nfig.savefig(out, dpi=200, bbox_inches="tight")\nout\n

### Rolling correlation (dependence is time-varying)\n

In [ ]:
window = 126  # ~6 months trading days\ntmp = rets.sort_values("Date").copy()\ntmp["rolling_corr"] = tmp["NFLX_logret"].rolling(window).corr(tmp["GLD_logret"])\n\nfig, ax = plt.subplots(figsize=(12, 4))\nax.plot(tmp["Date"], tmp["rolling_corr"], linewidth=1.5)\nax.axhline(0, color="black", linewidth=1, alpha=0.6)\nax.set_title(f"Rolling correlation (window={window} days)")\nax.set_ylabel("corr(NFLX, GLD)")\nfig.tight_layout()\nout = FIG_DIR / "rolling_corr_daily.png"\nfig.savefig(out, dpi=200, bbox_inches="tight")\nout\n

### Tail-day highlighting (worst 5% NFLX days)\n

In [ ]:
q = 0.05\ncutoff = rets["NFLX_logret"].quantile(q)\ntail = rets[rets["NFLX_logret"] <= cutoff].copy()\n\nfig, ax = plt.subplots(figsize=(7.5, 5.5))\nax.scatter(rets["GLD_logret"], rets["NFLX_logret"], s=10, alpha=0.15, label="All days")\nax.scatter(tail["GLD_logret"], tail["NFLX_logret"], s=18, alpha=0.6, label=f"Tail days (NFLX bottom {int(q*100)}%)")\nax.set_xlabel("GLD daily log return")\nax.set_ylabel("NFLX daily log return")\nax.set_title("Tail highlighting: dependence may differ in crashes")\nax.legend()\nfig.tight_layout()\nout = FIG_DIR / "tail_highlight_scatter_daily.png"\nfig.savefig(out, dpi=200, bbox_inches="tight")\nout\n